In [ ]:
from dataclasses import dataclass, replace
import functools
import pickle
from IPython.display import display
import ipywidgets as widgets
import cvxopt
import numpy as np
from matplotlib import pyplot
from donotation import do
import statemonad
import polymat, polymat.typing
import sosopt, sosopt.typing

# Initialize context object used to create concrete polynomials from polynomial expressions.
# The word context (instead of state) is used in order to not confuse it with the state of a dynamical system.
context = polymat.init_state()

# Finding Control Barrier Functions (CBFs) and Control Lyapunov Functions (CLFs) for an advanced Safety Filter

This notebook demonstrates how to set up a sum-of-squares (SOS) optimization problem to find a Control Barrier Function (CBF) for a safety filter applied to a dynamical system.
The CBF ensures that the system remains within an allowable set while respecting permissible ranges for stationary states.

The implementation relies on three Python packages:
- `statemonad`: For representing stateful operations.
- `polymat`: For polynomial matrix manipulations.
- `sosopt`: For solving SOS optimization problems.

## Problem setup

In the following, we define the system model, state and input variables, nominal controller, and constraints required to formulate the CBF and CLF synthesis problem.

### State and Input Variables

First, we define the state and input variables of the dynamical system. These are symbolic variables used to construct polynomial expressions.

In [ ]:
# Define a tuple of Python strings representing the state variables of the system model.
# Example: state_variable_names = ('x1', 'x2')
state_variable_names = ...  # [COMPLETE HERE]

# Create symbolic state variables using polymat
state_variables = tuple(polymat.define_variable(name) for name in state_variable_names)
x = polymat.v_stack(state_variables)
n_states = len(state_variables)

# Unpack the state variables into individual Python variables
# Example: x1, x2 = state_variables
... = state_variables  # [COMPLETE HERE]

# Define a tuple of Python strings representing the input variables of the system model.
# Example: input_variable_names = ('u1', 'u2')
input_variable_names = ...  # [COMPLETE HERE]

# Create symbolic input variables using polymat
input_variables = tuple(polymat.define_variable(name) for name in input_variable_names)
u = polymat.v_stack(input_variables)
n_inputs = len(input_variables)

# Unpack the input variables into individual Python variables
# Example: u1, u2 = input_variables
... = input_variables  # [COMPLETE HERE]

### System Model

The system dynamics are given by:

$$ \dot{x} = f(x) + G(x)u $$

where:
- $x$ is the state vector.
- $u$ is the input vector.
- $f(x)$ is the drift dynamics (an $n$-vector).
- $G(x)$ is the control matrix (an $n \times m$ matrix).

In [ ]:
@dataclass
class ModelParam:
    s_n: float
    w_n: float

    v_grid_phph: float
    r_grid_si: float
    l_grid_si: float

    r_tr: float
    x_tr: float

    c_dc_si: float
    r_dc_si: float
    v_dc_n: float

    def __post_init__(self):
        self.v_n = self.v_grid_phph / np.sqrt(3)
        self.i_n = self.s_n / self.v_n
        self.z_n = self.v_n**2 / self.s_n
        self.l_n = self.z_n / self.w_n
        self.z_dc_n = self.v_dc_n**2 / self.s_n
        self.c_dc_n = 1 / (self.z_dc_n * self.w_n)

        self.r_grid = self.r_grid_si / self.z_n
        self.x_grid = self.l_grid_si / self.l_n
        self.c_dc = self.c_dc_si / self.c_dc_n
        self.r_dc = self.r_dc_si / self.z_dc_n

        self.g_dc = 1 / self.r_dc

        self.l = self.x_tr + self.x_grid


model = ModelParam(
    s_n=20e6 / 3,
    w_n=2 * np.pi * 50,
    v_grid_phph=130e3,
    r_grid_si=1e-3,
    l_grid_si=0.01,
    r_tr=2 * 3e-3,
    x_tr=2 * 9e-2,
    c_dc_si=2 * 10e-3,
    r_dc_si=1000,
    v_dc_n=2400,
)

scale = polymat.from_((
    (1 / model.c_dc, 1 / model.l, 1 / model.l),
)).diag() * model.w_n

# Define a polymat n-vector representing f(x) using the state variables.
# Example:
# f = polymat.from_(((-x1,), (-x2,)))
f = ...  # [COMPLETE HERE]

# Define a polymat nxm matrix representing G(x) using the state variables.
# Example:
# G = polymat.from_(((1, 0), (0, 1)))
G = ...  # [COMPLETE HERE]

### Nominal Controller

The nominal controller $u_n(x)$ is a polynomial function of the state variables, representing the baseline control law before applying the safety filter.

In [ ]:
# Define a polymat m-vector representing the nominal controller using the state variables.
# Example:
# u_n = polymat.from_(((-x1,), (-x2,)))
u_n = ...  # [COMPLETE HERE]

# modify nominal controller in SOS constraints when no feasible solution can be found
u_n_sos = u_n

### Allowable Set and Permissible Ranges

The allowable set $X_a$ defines the safe region where the system should remain. 
It is represented by polynomial inequalities $w(x) \leq 0$.

The operating region $X_\text{op}$ defines the permissible range of stationary states considered for this problem. 
It is represented by polynomial inequalities $r(x) \leq 0$, and is selected to contain the allowable set $X_a$.

In [ ]:
i_max = 1.26  # todo: remove
vf_max = 1  # todo: remove
i_ref_max = 1.18  # todo: remove
i0_max = 0.6  # todo: remove

# Define a tuple of polymat polynomials representing the allowable set X_a.
# Example:
# w = {'w': x1**2 + x2**2 - 1}
w = ...  # [COMPLETE HERE]

# Define a tuple of polymat polynomials representing the permissible ranges of stationary states.
# Example:
# r = {}
r = ...  # [COMPLETE HERE]

### Polynomial Degrees

To formulate the SOS problem, we specify the degrees of the polynomials involved:
- `u_degrees`: Degrees for the control input corrections.
- `B_degrees`: Degrees for the CBF and CLF candidate.

In [ ]:
# Define the degrees for the CBF and CLF candidates.
# Example: degree = 4
degree = ...  # [COMPLETE HERE]

## Prerequisites

### CBF and CLF Candidates

We define CBFs for each polynomial in the allowable set `w` and a single CLF to ensure stability and safety. 
The constant coefficient of the CBFs are set to -1 to ensure numerical stability, and derivatives are computed for later use in SOS constraints.

In [ ]:
# Define one CBF for each polynomial in w
B_dict = {}
B_var_dict = {}  # For fixing parameters in the alternating algorithm
for name, expr in w.items():
    B_var_dict[name] = sosopt.define_polynomial(
        name=f"B_{name}", 
        monomials=x.combinations(degrees=range(1, degree + 1)),
    )
    B_dict[name] = B_var_dict[name] - 1
dB_dict = {name: B.diff(x).T.cache() for name, B in B_dict.items()}

# Define a single CLF
V_monom = x.combinations(degrees=range(degree + 1))
V = sosopt.define_polynomial(
    name="V", 
    monomials=V_monom,
)
dV = V.diff(x).T.cache()

### State Feedback Controller

A rational state feedback controller $u(x) = p(x) / s(x)$ is defined to ensure compatibility between CBFs and CLFs.
The numerator `p(x)` is a polynomial vector, and the denominator `s(x)` is a multiplier polynomial scaled to match the degree of the closed-loop system.

In [ ]:
# Define the numerator p(x) of the rational controller
p = sosopt.define_polynomial(
    name="p", 
    monomials=x.combinations(degrees=range(degree)),
    n_rows=n_inputs,
)

# Compute G(x) @ p(x)
G_at_p = (G @ p).cache()

# Compute the degree of the denominator s(x)
context, max_degrees = polymat.to_degree(G_at_p, variables=x).apply(context)

# Define the denominator s(x) as a multiplier
context, s = sosopt.define_multiplier(
    name="s",
    degree=int(np.max(max_degrees)),
    multiplicand=f,
    variables=x,
).apply(context)

# Compute s(x) * f(x)
s_f = (s * f).cache()

# Define closed-loop systems for different controllers
x_dot = (s_f + G_at_p).cache()          # With rational controller u(x) = p(x)/s(x)
x_dot_n = (s_f + G @ u_n).cache()       # With nominal controller u_n
x_dot_n_sos = (s_f + G @ u_n_sos).cache()  # With SOS-based nominal controller (define u_n_sos if needed)

### Operating Region

Two operating regions are defined—one for CBFs and one for the CLF—to improve the alternating algorithm's performance.
These regions are parameterized with decision variables `delta_opV` and `delta_opB`, initialized to 1 and minimized to 0 during optimization.

In [ ]:
# Define decision variables for operating region adjustment
delta_opV = sosopt.define_variable(name="delta_opV")
delta_opB = sosopt.define_variable(name="delta_opB")

# Define operating regions with adjustable sublevel sets
opV_dict = {name: op_region - delta_opV for name, op_region in r.items()}
opB_dict = {name: op_region - delta_opB for name, op_region in r.items()}

### Dissipation Rate

The dissipation rate ensures finite-time convergence to the nominal region. It is set to 0 here but can be adjusted based on system requirements.

In [ ]:
# Define the dissipation rate
dissipation_rate = 0  # Adjust as needed for convergence

## SOS Constraints

This section specifies the sum-of-squares (SOS) constraints that define the advanced safety filter.
These constraints ensure:
- Safety with respect to the safe set $\mathcal X_s$ via CBF conditions.
- Finite-time convergence to the nominal region $\mathcal X_n$ via CLF conditions.
- Containment of the nominal region $\mathcal X_n$ within the safety set $\mathcal X_s$, and the safety set $\mathcal X_s$ within the allowable region $\mathcal X_a$.
- Forward invariance of the nominal region $\mathcal X_n$ under the nominal controller $u_n(x)$.
- Positivity of the denominator and operating region adjustments.

An alternating algorithm will solve the resulting bilinear problem, using margins (`cbf_margin`, `clf_margin`) to ensure improvement of `delta_opV` and `delta_opB` in the consecutive iterations.

In [ ]:
# Margins for solving the bilinear problem using an alternating algorithm
cbf_margin_dict = {}
for name, B in B_dict.items():
    cbf_margin_dict[name] = sosopt.define_variable(name=f"cbf_margin_{name}")
clf_margin = sosopt.define_variable(name="clf_margin")

# Limit CLF/CBF condition margins for numerical stability
min_margin = -0.01

# Initialize tuple of constraints
constraints = tuple()

# Define constraints for each CBF
for name, B in B_dict.items():
    # Control Barrier Function (CBF) condition
    context, cbf_constraint = sosopt.sos_constraint_putinar(
        name=f"cbf_{name}",
        greater_than_zero=-(dB_dict[name].T @ x_dot) + cbf_margin_dict[name],
        domain=sosopt.set_(
            greater_than_zero=opB_dict,
            equal_zero={"B": B},
        ),
    ).apply(context)
    constraints = constraints + (cbf_constraint,)

    # CBF margin positivity constraint
    context, cbf_margin_constraint = sosopt.sos_constraint(
        name=f"cbf_margin_{name}_pos",
        greater_than_zero=cbf_margin_dict[name] - min_margin,
    ).apply(context)
    constraints = constraints + (cbf_margin_constraint,)
    
    # Ensure the safety set contains the nominal region
    context, v_in_b_constraint = sosopt.sos_constraint(
        name=f"v_in_b_{name}",
        greater_than_zero=V - B,
    ).apply(context)
    constraints = constraints + (v_in_b_constraint,)
    
    # Ensure the safety set is contained in the allowable region
    context, b_in_w_constraint = sosopt.sos_constraint_putinar(
        name=f"b_in_w_{name}",
        greater_than_zero=B,
        domain=sosopt.set_(
            greater_than_zero={"w": w[name]},
        ),
    ).apply(context)
    constraints = constraints + (b_in_w_constraint,)

# Control Lyapunov Function (CLF) condition
context, clf_condition = sosopt.sos_constraint_putinar(
    name="clf",
    greater_than_zero=-(dV.T @ x_dot) - dissipation_rate + clf_margin,
    domain=sosopt.set_(
        greater_than_zero={"V": V} | opV_dict,
    ),
).apply(context)
constraints = constraints + (clf_condition,)

# CLF margin positivity constraint
context, clf_margin_constraint = sosopt.sos_constraint(
    name="clf_margin_pos",
    greater_than_zero=clf_margin - min_margin,
).apply(context)
constraints = constraints + (clf_margin_constraint,)

# Forward invariance condition of the nominal region w.r.t. the nominal controller
context, clf_unom_condition = sosopt.sos_constraint_putinar(
    name="u_nom",
    greater_than_zero=-(dV.T @ x_dot_n_sos) - dissipation_rate,
    domain=sosopt.set_(
        greater_than_zero=opV_dict,
        equal_zero={"V": V},
    ),
).apply(context)
constraints = constraints + (clf_unom_condition,)

# Positivity condition of the denominator
context, s_pos_constraint = sosopt.sos_constraint(
    name="s_pos",
    greater_than_zero=s - 0.001,
).apply(context)
constraints = constraints + (s_pos_constraint,)

# Positivity condition for operating region adjustments
context, delta_opV_constraint = sosopt.sos_constraint(
    name="delta_opV_pos",
    greater_than_zero=delta_opV,
).apply(context)
constraints = constraints + (delta_opV_constraint,)

context, delta_opB_constraint = sosopt.sos_constraint(
    name="delta_opB_pos",
    greater_than_zero=delta_opB,
).apply(context)
constraints = constraints + (delta_opB_constraint,)

# Convert constraints to a dictionary for easier reference
constraints = {c.name: c for c in constraints}

## Initialization

This section generates an initial set of values to initialize the alternating algorithm used to solve the SOS optimization problem. The initialization can either be computed programmatically or loaded from a pickle file. The initial values include:
- The numerator polynomial `p` of the state feedback controller.
- The denominator scalar `s`.
- Multipliers introduced through Putinar's Positivstellensatz.
- Operating region adjustments `delta_opV` and `delta_opB`.

These values are stored in a dictionary with the associated symbol of the variable.

In [ ]:
# Import additional required library for file loading
import pickle

# If True, load the initialization from a pickle file
load_from_file = False

if not load_from_file:
    # Initial numerator polynomial of the state feedback controller
    p0 = (polymat.from_(-np.ones((n_inputs, n_states))) @ x)
    
    # Initialize the algorithm with an initial guess
    init_values = (
        (p, p0),
        (s, 100),
        (constraints['clf'].multipliers["V"], 100),
        (constraints['u_nom'].multipliers["V"], 100),
        *(
            (constraints[f"cbf_{name}"].multipliers["B"], 100) for name in B_dict
        ),
        (delta_opV, 1),
        (delta_opB, 1),
    )

    initial_symbol_values = {}

    for expr, value_expr in init_values:
        if isinstance(value_expr, (float, int)):
            value_expr = polymat.from_vector(value_expr)
        
        match expr:
            case sosopt.typing.PolynomialVariable(monomials=monomials):
                for (row, col), param in expr.iterate_coefficients():
                    context, (data, *_) = polymat.to_tuple(
                        value_expr[row, col].linear_in(variables=x, monomials=monomials)
                    ).apply(context)
                    
                    initial_symbol_values[param.symbol] = data
    
            case polymat.typing.VariableExpression() as var_expr:
                context, (data, *_) = polymat.to_tuple(
                    value_expr.linear_in(variables=x)
                ).apply(context)
                
                initial_symbol_values[var_expr.symbol] = data

else:
    file_name = ...  # [COMPLETE HERE] e.g., 'initial_values.pkl'
    with open(file_name, 'rb') as file:   
        initial_symbol_values = pickle.load(file)

## Alternating Algorithm

This section defines and implements the alternating algorithm to solve the bilinear SOS optimization problem for CBF/CLF synthesis. The algorithm alternates between optimizing different sets of variables (e.g., controller parameters vs. CBF/CLF polynomials) to enforce safety and stability constraints. It includes:
- Data collection for each iteration.
- Definition of optimization steps with linear/quadratic costs and substitutions.
- Solver selection and configuration.
- Two phases: initialization (to reduce operating region margins) and alternation (to maximize the safe set volume).

### Data Collected at Each Iteration

We define a data class to store the state, symbol values, and solver results for each iteration.

In [ ]:
from dataclasses import dataclass, replace

@dataclass
class IterationData:
    context: polymat.typing.State
    symbol_values: dict
    solver_data: sosopt.typing.SolverData | None

iter_data = IterationData(context=context, symbol_values=initial_symbol_values, solver_data=None)

### Define Steps/Alternations of the Algorithm

We define a `Step` class to encapsulate each optimization step, including costs, substitutions, and overrides. Multiple steps are tailored to optimize margins, operating regions, and safe set volume.

In [ ]:
@dataclass
class Step:
    lin_cost: polymat.typing.MatrixExpression
    quad_cost: polymat.typing.MatrixExpression | None
    substitutions: tuple
    override_symbol_values: dict

def init_step(lin_cost, substitutions, quad_cost=None, override_symbol_values={}):
    return Step(
        lin_cost=lin_cost, quad_cost=quad_cost, 
        substitutions=substitutions, 
        override_symbol_values=override_symbol_values,
    )

# Set CLF/CBF margins to zero when they are not being optimized
set_epsilon_to_zero = {
    clf_margin.symbol: (0,),
    **{cbf_margin.symbol: (0,) for cbf_margin in cbf_margin_dict.values()}
}

step_1_substitutions = (
    p, 
    s,
    constraints['clf'].multipliers["V"],
    constraints['u_nom'].multipliers["V"],
    *(constraints[f'cbf_{name}'].multipliers["B"] for name in B_dict)
)

step_2_substitutions = (
    V,
    *B_var_dict.values()
)

step_1_margin = init_step(
    lin_cost=clf_margin + sum(cbf_margin_dict.values()),
    substitutions=step_1_substitutions + (
        delta_opV, 
        delta_opB,
    )
)

step_1_roiV = init_step(
    lin_cost=delta_opV,
    substitutions=step_1_substitutions + (
        delta_opB, 
        *(constraints['clf'].multipliers[r_name] for r_name in r),
        *(constraints['u_nom'].multipliers[r_name] for r_name in r),
        clf_margin, 
        *cbf_margin_dict.values(),
    ),
    override_symbol_values=set_epsilon_to_zero,
)

step_1_roiB = init_step(
    lin_cost=delta_opB,
    substitutions=step_1_substitutions + (
        delta_opV,
        *(constraints[f'cbf_{name}'].multipliers[r_name] for name in B_dict for r_name in r),
        clf_margin,
        *cbf_margin_dict.values(),
    ),
    override_symbol_values=set_epsilon_to_zero,
)

step_1_vol = init_step(
    lin_cost=sum(B.quadratic_in(x).trace() for B in B_dict.values()),
    substitutions=step_1_substitutions + (
        delta_opV, 
        delta_opB,
        clf_margin, 
        *cbf_margin_dict.values(),
    ),
    override_symbol_values=set_epsilon_to_zero,
)

step_2_margin = init_step(
    lin_cost=clf_margin + sum(cbf_margin_dict.values()),
    substitutions=step_2_substitutions + (
        delta_opV, 
        delta_opB,
    )
)

step_2_roiV = init_step(
    lin_cost=delta_opV,
    substitutions=step_2_substitutions + (
        delta_opB,
        *(constraints['clf'].multipliers[r_name] for r_name in r),
        *(constraints['u_nom'].multipliers[r_name] for r_name in r),
        clf_margin, 
        *cbf_margin_dict.values(),
    ),
    override_symbol_values=set_epsilon_to_zero,
)

step_2_roiB = init_step(
    lin_cost=delta_opB,
    substitutions=step_2_substitutions + (
        delta_opV,
        *(constraints[f'cbf_{name}'].multipliers[r_name] for name in B_dict for r_name in r),
        clf_margin, 
        *cbf_margin_dict.values(),
    ),
    override_symbol_values=set_epsilon_to_zero,
)

### Select Solver

We select the CVXOPT solver (with an option for MOSEK) and configure its options to suppress progress output and set iteration limits.

In [ ]:
import cvxopt

# Select solver
solver = sosopt.cvx_opt_solver
# solver = sosopt.mosek_solver  # Uncomment to use MOSEK instead

# CVXOPT solver options
cvxopt.solvers.options['show_progress'] = False
# cvxopt.solvers.options['show_progress'] = True  # Uncomment for verbose output
# cvxopt.solvers.options['maxiters'] = 100  # Uncomment to limit iterations

### Solve Problem Function

A utility function is defined to solve the SOS problem for each step, updating iteration data with solver results.

In [ ]:
def solve_problem(context, step: Step, iter_data: IterationData):
    problem = sosopt.sos_problem(
        lin_cost=step.lin_cost,
        quad_cost=step.quad_cost,
        constraints=constraints.values(),
        solver=solver,
    )

    # Overwrite values
    symbol_values = iter_data.symbol_values | step.override_symbol_values

    substitutions = {
        symbol: symbol_values[symbol] for param in step.substitutions for symbol in param.iterate_symbols()
    }
    
    # Filter values that need to be substituted
    problem = problem.eval(substitutions)

    # # Uncomment to print decision variables for each constraint
    # for primitive in problem.constraint_primitives:
    #     print(f'{primitive.name=}, {primitive.decision_variable_symbols=}')
    
    # Solve SOS problem
    context, sos_result = problem.solve().apply(context)

    solver_data = sos_result.solver_data
    print(f'{solver_data.status=}, {solver_data.iterations=}, {solver_data.cost=}')
    
    # Update iteration data
    n_iter_data = replace(
        iter_data, 
        symbol_values=symbol_values | sos_result.symbol_values, 
        solver_data=solver_data,
    )
    
    epsilon_roi = (clf_margin, *cbf_margin_dict.values(), delta_opV, delta_opB)
    print(f'epsilon = {tuple(n_iter_data.symbol_values[e.symbol] for e in epsilon_roi)}')

    return context, n_iter_data

### Run Initialization Algorithm

The initialization phase reduces the operating region margins (`delta_opV`, `delta_opB`) below a threshold (1e-6) by alternating optimization steps.

In [ ]:
for idx in range(20):
    print(f'iteration: {idx}')
        
    if 1e-6 < iter_data.symbol_values[delta_opB.symbol][0]:
        context, iter_data = solve_problem(context, step_1_margin, iter_data)
        context, iter_data = solve_problem(context, step_1_roiB, iter_data)
    
    if 1e-6 < iter_data.symbol_values[delta_opV.symbol][0]:
        context, iter_data = solve_problem(context, step_2_margin, iter_data)
        context, iter_data = solve_problem(context, step_2_roiV, iter_data)

        context, iter_data = solve_problem(context, step_1_margin, iter_data)
        context, iter_data = solve_problem(context, step_1_roiV, iter_data)
    else:
        break
    
    if 1e-6 < iter_data.symbol_values[delta_opB.symbol][0]:
        context, iter_data = solve_problem(context, step_2_margin, iter_data)
        context, iter_data = solve_problem(context, step_2_roiB, iter_data)

### Run Alternating Algorithm

After initialization, the main algorithm maximizes a surrogate volume of the safe set (via `step_1_vol`) until the primal cost decreases by less than 1%.

In [ ]:
# Maximize a surrogate volume of the safe set until the primal cost decreases by less than 1 percent
previous_cost = None

if not (
    1e-6 < iter_data.symbol_values[delta_opV.symbol][0]
    or 1e-6 < iter_data.symbol_values[delta_opB.symbol][0]
):
    for idx in range(20):
        print(f'iteration: {idx}')
    
        context, iter_data = solve_problem(context, step_2_margin, iter_data)
        context, iter_data = solve_problem(context, step_1_vol, iter_data)

        if previous_cost is not None:
            if (previous_cost - iter_data.solver_data.cost) < 0.01 * iter_data.solver_data.cost:
                break
        previous_cost = iter_data.solver_data.cost

In [ ]:
def save_symbol_values(arg):
    file_name = '2_symbol_values.p'
    
    with open(file_name, 'wb') as file:   
        pickle.dump(iter_data.symbol_values, file)

# Create a button to ensure that the file is not overritten by accident.
button_download = widgets.Button(description = 'Save symbol values')   
button_download.on_click(save_symbol_values)
display(button_download)

In [ ]:
@do()
def debugging_tools():

    # select the function to be evaluated
    expr = constraints['clf'].multipliers["V"]
    # expr = V

    func = yield polymat.to_array(expr.eval(iter_data.symbol_values), x)
    x = np.array((0.1, 0, 0)).reshape(-1, 1)

    # evaluates the function at x
    print(f'{func(x)=}')

    return statemonad.from_(None)

_ = debugging_tools().apply(context)

## Plot Results

This section visualizes the results of the CBF/CLF synthesis by plotting:
- A stream plot of the closed-loop system dynamics under the optimized controller.
- Sublevel sets of the Control Lyapunov Function (CLF) `V` and Control Barrier Functions (CBFs) `B_dict`.

The plot is generated for a 2D projection of the state space, assuming the first two dimensions are of interest (e.g., `x1` and `x2`). Adjust the variable labels and limits as needed for your specific system.

In [ ]:
import matplotlib.pyplot as pyplot
import numpy.matlib
import numpy as np

def plot_result(context, symbol_values):
    # Helper function to project the 3-dimensional state onto 2 dimensions
    def map_to_xy(x, y):
        return np.array((x, y) + (0,) * (n_states - 2)).reshape(-1, 1)
    
    pyplot.close()
    fig = pyplot.figure(figsize=(8, 8))
    ax = fig.subplots()
    
    # Create boundary box
    ####################
    x_min, x_max, y_min, y_max = -0.8, 0.2, -1.3, 1.3
    args = {'color': 'r', 'linestyle': 'dashed', 'dashes': (5, 5), 'linewidth': 0.8}
    ax.plot(np.array((x_min, x_max)), np.array((y_min, y_min)), **args)
    ax.plot(np.array((x_min, x_max)), np.array((y_max, y_max)), **args)
    ax.plot(np.array((x_min, x_min)), np.array((y_min, y_max)), **args)
    ax.plot(np.array((x_max, x_max)), np.array((y_min, y_max)), **args)
    
    # Create stream plot
    ####################
    context, f_array = polymat.to_array(f, x).apply(context)
    context, G_array = polymat.to_array(G, x).apply(context)
    context, p_array = polymat.to_array(p.eval(symbol_values), x).apply(context)
    context, s_array = polymat.to_array(s.eval(symbol_values), x).apply(context)
    
    def get_x_dot(x):
        x = np.array(x).reshape(-1, 1)
        u = p_array(x) / s_array(x)
        xdot = f_array(x) + G_array(x) @ u
        return np.squeeze(xdot)
    
    ticksX = np.arange(-0.8, 0.2, 0.04)
    ticksY = np.arange(-1.3, 1.34, 0.04)
    n_row, n_col = len(ticksY), len(ticksX)
    X = np.matlib.repmat(ticksX, n_row, 1)
    Y = np.matlib.repmat(ticksY.reshape(-1, 1), 1, n_col)
    
    stream_U = np.zeros((n_row, n_col))
    stream_V = np.zeros((n_row, n_col))
    def create_stream_data():
        for row, (x_row, y_row) in enumerate(zip(X, Y)):
            for col, (x, y_val) in enumerate(zip(x_row, y_row)):
                u, v, *_ = get_x_dot(map_to_xy(x, y_val))
                stream_U[row, col] = u
                stream_V[row, col] = v
    
    create_stream_data()
    ax.streamplot(X, Y, stream_U, stream_V, density=[0.4, 0.7])
    
    # Plot Sublevel sets
    ####################
    ticks = np.arange(-2.1, 2.1, 0.04)
    X = np.matlib.repmat(ticks, len(ticks), 1)
    Y = X.T
    
    context, V_array = polymat.to_array(V.eval(symbol_values), x).apply(context)
    ZV = np.vectorize(lambda x, y: V_array(map_to_xy(x, y)))(X, Y)
    ax.contour(X, Y, ZV, [0.0], linewidths=2, colors=['#17202A'])
    
    context, B1_array = polymat.to_array(B_dict['w1'].eval(symbol_values), x).apply(context)
    ZB1 = np.vectorize(lambda x, y: B1_array(map_to_xy(x, y)))(X, Y)
    ax.contour(X, Y, ZB1, [0.0], linewidths=0.5, colors=['#A0B1BA'])
    
    context, B2_array = polymat.to_array(B_dict['w2'].eval(symbol_values), x).apply(context)
    ZB2 = np.vectorize(lambda x, y: B2_array(map_to_xy(x, y)))(X, Y)
    ax.contour(X, Y, ZB2, [0.0], linewidths=0.5, colors=['#A0B1BA'])
    
    def select_greater(x, y):   
        v1 = B1_array(map_to_xy(x, y))
        v2 = B2_array(map_to_xy(x, y))
        return v2 if v1 < v2 else v1
        
    Zpick = np.vectorize(select_greater)(X, Y)
    CS = ax.contour(X, Y, Zpick, levels=[0], linewidths=2, colors=['#17202A'])
    
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    
    ax.set_xlabel(r'${\tilde v}_{dc}$ [p.u.]')
    ax.set_ylabel(r'${\tilde i}_{d}$ [p.u.]')
    
    pyplot.show()
    return context, fig

iter_data.context, fig = plot_result(iter_data.context, iter_data.symbol_values)

### Save Figure to PDF File

This section provides an interactive button to save the generated figure as a PDF file, preventing accidental overwrites.

In [ ]:
def save_figure(arg):
    fig.savefig('2_stream_plot.pdf', bbox_inches='tight')

# Create a button to ensure the file is not overwritten by accident
button_download = widgets.Button(description='Save figure')   
button_download.on_click(save_figure)
display(button_download)

## Pickle Results

This subsection saves the results in a pickle file, enabling further simulations and analysis.

In [ ]:
def save_arrays(arg):
    @do()
    def gen_arrays(symbol_values):
        symbol_values = iter_data.symbol_values

        s_array = yield from polymat.to_array(s.eval(symbol_values), x)
        
        V_array = yield from polymat.to_array(V.eval(symbol_values), x)
        B1_array = yield from polymat.to_array(B1.eval(symbol_values), x)
        B2_array = yield from polymat.to_array(B2.eval(symbol_values), x)

        dV_array = yield from polymat.to_array(dV.eval(symbol_values), x)
        dB1_array = yield from polymat.to_array(dB1.eval(symbol_values), x)
        dB2_array = yield from polymat.to_array(dB2.eval(symbol_values), x)

        gV_array = yield from polymat.to_array(constraints['clf'].multipliers['V'].eval(symbol_values), x)
        gB1_array = yield from polymat.to_array(constraints['cbf1'].multipliers['B'].eval(symbol_values), x)
        gB2_array = yield from polymat.to_array(constraints['cbf2'].multipliers['B'].eval(symbol_values), x)

        arrays = {
            's': s_array,
            'V': V_array,
            'B1': B1_array,
            'B2': B2_array,
            'dV': dV_array,
            'dB1': dB1_array,
            'dB2': dB2_array,
            'gV': gV_array,
            'gB1': gB1_array,
            'gB2': gB2_array,
        }

        return statemonad.from_(arrays)

    _, arrays = gen_arrays(iter_data.symbol_values).apply(context)

    file_name = '2_arrays.p'

    with open(file_name, 'wb') as file:   
        pickle.dump(arrays, file)

# Create a button to ensure that the file is not overritten by accident.
button_download = widgets.Button(description = 'Save arrays')   
button_download.on_click(save_arrays)
display(button_download)